### **Import Libraries**

In [1]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    Imputer,
    VectorAssembler,
    StandardScaler
)

### **Initialize Spark Session**

In [2]:
spark = (
    SparkSession.builder
    .appName("EEG_Data_Preprocessing")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/13 08:23:47 WARN Utils: Your hostname, Sarras-MacBook-Air-2.local, resolves to a loopback address: 127.0.0.1; using 192.168.234.219 instead (on interface en0)
26/04/13 08:23:47 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/13 08:23:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### **Define Paths**

In [3]:
project_root = Path("..").resolve()

input_dir = project_root / "data" / "processed" / "feature_engineered" / "final_modeling_table"
demographic_path = project_root / "data" / "demographic.csv"

output_dir = project_root / "data" / "processed" / "preprocessed_final_modeling_table"
pipeline_dir = project_root / "data" / "processed" / "preprocessing_pipeline"

output_dir.mkdir(parents=True, exist_ok=True)
pipeline_dir.mkdir(parents=True, exist_ok=True)

### **Load the Engineered Modeling Table**

In [10]:
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(input_dir))
)

print(f"Number of columns in modeling table: {len(df.columns)}")
df.select("subject", "trial", "condition").show(5, truncate=False)

Number of columns in modeling table: 174
+-------+-----+---------+
|subject|trial|condition|
+-------+-----+---------+
|49.0   |1.0  |1.0      |
|49.0   |1.0  |2.0      |
|49.0   |1.0  |3.0      |
|49.0   |2.0  |1.0      |
|49.0   |2.0  |2.0      |
+-------+-----+---------+
only showing top 5 rows


### **Load the Demographic Table**

In [11]:
demographic_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(demographic_path))
)

demographic_df.show(5, truncate=False)

+-------+------+-------+----+----------+
|subject| group| gender| age| education|
+-------+------+-------+----+----------+
|1      |0.0   | M     |44.0|16.0      |
|2      |0.0   | M     |39.0|17.0      |
|3      |0.0   | M     |53.0|18.0      |
|4      |0.0   | M     |52.0|15.0      |
|5      |0.0   | M     |41.0|16.0      |
+-------+------+-------+----+----------+
only showing top 5 rows


### **Remove Any Duplicate Rows**

In [12]:
df = df.dropDuplicates(["subject", "trial", "condition"])

### **Clean and Prepare the Target Labels**

In [14]:
print(demographic_df.columns)

['subject', ' group', ' gender', ' age', ' education']


In [15]:
demographic_df = demographic_df.dropDuplicates(["subject"])

demographic_df = demographic_df.withColumnRenamed(" group", "label")

demographic_df = demographic_df.select("subject", "label")

### **Join the Target Labels to the Trial-Level Table**

In [17]:
df = df.join(
    demographic_df,
    on="subject",
    how="left"
)

### **Validate the Target Join**

In [18]:
df.select(
    "subject",
    "trial",
    "condition",
    "label"
).show(10, truncate=False)

+-------+-----+---------+-----+
|subject|trial|condition|label|
+-------+-----+---------+-----+
|49.0   |19.0 |1.0      |1.0  |
|47.0   |6.0  |3.0      |1.0  |
|47.0   |10.0 |3.0      |1.0  |
|42.0   |19.0 |1.0      |1.0  |
|44.0   |52.0 |1.0      |1.0  |
|44.0   |70.0 |1.0      |1.0  |
|45.0   |24.0 |2.0      |1.0  |
|45.0   |65.0 |2.0      |1.0  |
|46.0   |6.0  |1.0      |1.0  |
|46.0   |31.0 |1.0      |1.0  |
+-------+-----+---------+-----+
only showing top 10 rows


### **Identify Missing Values per Column**

In [19]:
missing_summary = df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

missing_summary.show(truncate=False)

26/04/13 08:31:44 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+-----+---------+---------+---------+-----------+-----------+----------+----------+------------+------------+---------+---------+-----------+-----------+----------+----------+------------+------------+----------+----------+------------+------------+---------+---------+-----------+-----------+---------+---------+-----------+-----------+----------+----------+------------+------------+----------+----------+------------+------------+--------------+--------------+--------------------+---------------+-----------+------------+-------------+--------------+--------------+--------------------+---------------+-----------+------------+-------------+----------------+----------------+----------------------+-----------------+-------------+--------------+---------------+----------------+----------------+----------------------+-----------------+-------------+--------------+---------------+------------------+------------------+-------------------+-------------------+------------------+----------

### **Define Identifier, Target, Categorical, and Numeric Columns**

In [20]:
id_cols = ["subject", "trial"]
target_col = "label"
categorical_cols = ["condition"]

excluded_cols = set(id_cols + [target_col] + categorical_cols)

numeric_cols = [
    field.name
    for field in df.schema.fields
    if field.name not in excluded_cols
    and isinstance(
        field.dataType,
        (
            T.IntegerType,
            T.LongType,
            T.FloatType,
            T.DoubleType,
            T.ShortType,
            T.DecimalType
        )
    )
]

print(f"Categorical columns: {categorical_cols}")
print(f"Numeric columns: {len(numeric_cols)}")

Categorical columns: ['condition']
Numeric columns: 171


### **Convert Categorical Columns to String Type**

In [22]:
for c in categorical_cols:
    df = df.withColumn(c, F.col(c).cast("string"))

### **Define the Categorical Encoding Stage**

In [23]:
indexer_stages = [
    StringIndexer(
        inputCol=col_name,
        outputCol=f"{col_name}_indexed",
        handleInvalid="keep"
    )
    for col_name in categorical_cols
]

### **Define the One-Hot Encoding Stage**

In [24]:
encoded_input_cols = [f"{c}_indexed" for c in categorical_cols]
encoded_output_cols = [f"{c}_ohe" for c in categorical_cols]

encoder_stage = OneHotEncoder(
    inputCols=encoded_input_cols,
    outputCols=encoded_output_cols,
    handleInvalid="keep"
)

### **Define the Feature Assembly Stage**

In [25]:
feature_input_cols = numeric_cols + encoded_output_cols

assembler_stage = VectorAssembler(
    inputCols=feature_input_cols,
    outputCol="features_raw",
    handleInvalid="skip"
)

### **Define the Feature Scaling Stage**

In [26]:
scaler_stage = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=False,
    withStd=True
)

### **Build the Complete Preprocessing Pipeline**

In [27]:
preprocessing_pipeline = Pipeline(
    stages=indexer_stages + [
        encoder_stage,
        assembler_stage,
        scaler_stage
    ]
)

### **Fit the Preprocessing Pipeline and Transform the Dataset**

In [28]:
preprocessing_model = preprocessing_pipeline.fit(df)
preprocessed_df = preprocessing_model.transform(df)

### **Review the Preprocessed Output**

In [29]:
preprocessed_df.select(
    "subject",
    "trial",
    "condition",
    "label",
    "features_raw",
    "features"
).show(5, truncate=False)

+-------+-----+---------+-----+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

### **Build the Final Preprocessed Modeling Table**

In [30]:
final_preprocessed_df = preprocessed_df.select(
    "subject",
    "trial",
    "condition",
    "label",
    "features"
)

In [31]:
final_preprocessed_df.select(
    "subject",
    "trial",
    "condition",
    "label"
).show(10, truncate=False)

+-------+-----+---------+-----+
|subject|trial|condition|label|
+-------+-----+---------+-----+
|49.0   |19.0 |1.0      |1.0  |
|47.0   |6.0  |3.0      |1.0  |
|47.0   |10.0 |3.0      |1.0  |
|42.0   |19.0 |1.0      |1.0  |
|44.0   |52.0 |1.0      |1.0  |
|44.0   |70.0 |1.0      |1.0  |
|45.0   |24.0 |2.0      |1.0  |
|45.0   |65.0 |2.0      |1.0  |
|46.0   |6.0  |1.0      |1.0  |
|46.0   |31.0 |1.0      |1.0  |
+-------+-----+---------+-----+
only showing top 10 rows


### **Save the Final Preprocessed Dataset**

In [32]:
(
    final_preprocessed_df
    .write
    .mode("overwrite")
    .parquet(str(output_dir))
)

26/04/13 08:50:53 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/13 08:50:53 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/04/13 08:50:53 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/04/13 08:50:55 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/04/13 08:50:55 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


### **Save the Fitted Preprocessing Pipeline**

In [34]:
preprocessing_model.write().overwrite().save(str(pipeline_dir))